In [ ]:

import networkx as nx
import numpy as np

def contar_ciclos_cerrados(nodos, aristas):
    """
    Cuenta el número de ciclos cerrados (caminos que vuelven al inicio)
    en el grafo del clúster.
    """
    G = nx.Graph()
    G.add_nodes_from(range(len(nodos)))
    G.add_edges_from(aristas)

    # Número de ciclos cerrados de longitud L para L = 3, 4, 5...
    # Usando la traza de A^L / 2L
    A = nx.to_numpy_array(G)
    ciclos = {}
    for L in range(3, 12):
        AL = np.linalg.matrix_power(A, L)
        ciclos[L] = int(np.trace(AL) // (2 * L))
    return ciclos, G

# Clúster 19
nodos_19 = [(0,0)]
nodos_19 += [(1,0),(1,-1),(0,-1),(-1,0),(-1,1),(0,1)]
nodos_19 += [(2,0),(2,-1),(2,-2),(1,-2),(0,-2),(-1,-1),
             (-2,0),(-2,1),(-2,2),(-1,2),(0,2),(1,1)]

vecinos = [(1,0),(-1,0),(0,1),(0,-1),(1,-1),(-1,1)]
nodo_set = set(nodos_19)
aristas_19 = []
for i, (q, r) in enumerate(nodos_19):
    for dq, dr in vecinos:
        if (q+dq, r+dr) in nodo_set:
            j = nodos_19.index((q+dq, r+dr))
            if i < j:
                aristas_19.append((i, j))

ciclos_19, G19 = contar_ciclos_cerrados(nodos_19, aristas_19)

print(f"=== CLÚSTER 19 ===")
print(f"Nodos: {len(nodos_19)}")
print(f"Aristas: {len(aristas_19)}")
print(f"Ciclos cerrados por longitud:")
total_19 = 0
for L, n in ciclos_19.items():
    print(f"  L={L}: {n}")
    total_19 += n
print(f"Total de ciclos cerrados: {total_19}")

# Clúster 57 (Hard-Lock)
# Construcción: 3 clústeres 19 + aristas de acoplamiento
nodos_57 = []
aristas_57 = []
for k in range(3):
    offset = k * 19
    for (q, r) in nodos_19:
        # Rotación de 120° = offset en coordenadas
        nodos_57.append((q + k*10, r))  # separación
    for (i, j) in aristas_19:
        aristas_57.append((i + offset, j + offset))

# Aristas de acoplamiento entre clústeres
aristas_57.append((0, 19))
aristas_57.append((19, 38))
aristas_57.append((38, 0))

ciclos_57, G57 = contar_ciclos_cerrados(nodos_57, aristas_57)

print(f"\n=== HARD-LOCK 57 ===")
print(f"Nodos: {len(nodos_57)}")
print(f"Aristas: {len(aristas_57)}")
total_57 = 0
for L, n in ciclos_57.items():
    if n > 0:
        print(f"  L={L}: {n}")
        total_57 += n
print(f"Total de ciclos cerrados: {total_57}")

# Ratio
if total_19 > 0:
    print(f"\nRatio ciclos: {total_57}/{total_19} = {total_57/total_19:.3f}")
    print(f"Ratio masas (objetivo): 1836.15")

In [ ]:

"""
AA Search: búsqueda de invariantes topológicos que reproduzcan
las relaciones de masa observadas en física de partículas.

Autor: Edward P. López (con asistencia de BRO)
Fecha: 2026-09-18
"""

import numpy as np
import networkx as nx
import itertools
from scipy import stats
from scipy.linalg import eigvalsh
import json

# ============================================================
# DATOS EXTERNOS (CODATA 2022, PDG 2022)
# ============================================================
RATIOS_OBSERVADOS = {
    'm_p/m_e':     1836.152673426,
    'm_mu/m_e':    206.7682830,
    'm_tau/m_e':   3477.228,
    'm_n/m_p':     1.00137841930,
    'm_mu/m_p':    0.1126095264,
    'm_tau/m_p':   1.893763,
}

# ============================================================
# CONSTRUCCIÓN DE LOS CLÚSTERES
# ============================================================
def build_cluster_19():
    """Clúster 19 hexagonal centrado."""
    nodos = [(0,0)]
    nodos += [(1,0),(1,-1),(0,-1),(-1,0),(-1,1),(0,1)]
    nodos += [(2,0),(2,-1),(2,-2),(1,-2),(0,-2),(-1,-1),
              (-2,0),(-2,1),(-2,2),(-1,2),(0,2),(1,1)]
    vecinos = [(1,0),(-1,0),(0,1),(0,-1),(1,-1),(-1,1)]
    nodo_set = set(nodos)
    aristas = []
    for i, (q, r) in enumerate(nodos):
        for dq, dr in vecinos:
            if (q+dq, r+dr) in nodo_set:
                j = nodos.index((q+dq, r+dr))
                if i < j:
                    aristas.append((i, j))
    G = nx.Graph()
    G.add_nodes_from(range(len(nodos)))
    G.add_edges_from(aristas)
    return G

def build_cluster_57():
    """Hard-Lock 57: 3 clústeres 19 acoplados."""
    G19 = build_cluster_19()
    G = nx.Graph()
    # 3 copias del clúster 19
    for k in range(3):
        offset = k * 19
        for (u, v) in G19.edges():
            G.add_edge(u + offset, v + offset)
    # Acoplamiento: nodos centrales en anillo
    G.add_edge(0, 19)    # centro 1 - centro 2
    G.add_edge(19, 38)   # centro 2 - centro 3
    G.add_edge(38, 0)    # centro 3 - centro 1
    return G

# ============================================================
# INVARIANTES DE GRAFO
# ============================================================
def invariantes_basicos(G):
    """Invariantes topológicos elementales."""
    N = G.number_of_nodes()
    E = G.number_of_edges()
    dim_ciclos = E - N + 1  # dimension del espacio de ciclos
    return {
        'N': N,
        'E': E,
        'dim_cycle': dim_ciclos,
        'densidad': 2*E / (N*(N-1)),
    }

def ciclos_por_longitud(G, L_max=12):
    """Número de ciclos cerrados por longitud."""
    A = nx.to_numpy_array(G)
    ciclos = {}
    for L in range(3, L_max+1):
        AL = np.linalg.matrix_power(A, L)
        ciclos[L] = int(np.trace(AL) // (2*L))
    return ciclos

def total_ciclos(G, L_max=12):
    """Total de ciclos cerrados hasta L_max."""
    return sum(ciclos_por_longitud(G, L_max).values())

def spanning_trees(G):
    """Número de árboles de expansión (Kirchhoff)."""
    L = nx.laplacian_matrix(G).toarray()
    # Remover primera fila y columna
    L_red = L[1:, 1:]
    return int(round(np.linalg.det(L_red)))

def espectro_laplaciano(G):
    """Espectro completo del Laplaciano."""
    L = nx.laplacian_matrix(G).toarray().astype(float)
    eigs = np.sort(eigvalsh(L))
    return eigs

def invariantes_espectrales(G):
    """Invariantes basados en el espectro del Laplaciano."""
    eigs = espectro_laplaciano(G)
    eigs_nz = eigs[eigs > 1e-10]  # no cero
    return {
        'lambda_max': float(eigs[-1]),
        'lambda_1': float(eigs_nz[0]) if len(eigs_nz) > 0 else 0,
        'lambda_2': float(eigs_nz[1]) if len(eigs_nz) > 1 else 0,
        'spectral_gap': float(eigs_nz[0]) if len(eigs_nz) > 0 else 0,
        'sum_eigs': float(np.sum(eigs_nz)),
        'prod_eigs_nonzero': float(np.prod(eigs_nz)),
        'radius_spectral': float(np.max(np.abs(eigs))),
        'energia_total': float(np.sum(eigs**2)),
    }

def invariantes_espectrales_normalizados(G):
    """Invariantes del Laplaciano normalizado."""
    L = nx.laplacian_matrix(G).toarray().astype(float)
    D = np.diag(np.sum(L, axis=1))
    D_inv_sqrt = np.diag(1/np.sqrt(np.diag(D)))
    L_norm = np.eye(L.shape[0]) - D_inv_sqrt @ (D - L) @ D_inv_sqrt
    eigs = np.sort(eigvalsh(L_norm))
    eigs_nz = eigs[eigs > 1e-10]
    return {
        'lambda1_norm': float(eigs_nz[0]) if len(eigs_nz) > 0 else 0,
        'lambda_max_norm': float(eigs[-1]),
        'spectral_gap_norm': float(eigs_nz[0]) if len(eigs_nz) > 0 else 0,
    }

def caminos_mas_cortos(G):
    """Distancias topológicas promedio."""
    N = G.number_of_nodes()
    dists = dict(nx.all_pairs_shortest_path_length(G))
    diam = nx.diameter(G)
    avg_path = nx.average_shortest_path_length(G)
    return {
        'diametro': diam,
        'camino_promedio': avg_path,
        'dist_total': sum(sum(d.values()) for d in dists.values()) // 2,
    }

def autovalores_adyacencia(G):
    """Autovalores de la matriz de adyacencia."""
    A = nx.to_numpy_array(G)
    eigs = np.sort(eigvalsh(A))
    return {
        'adj_max': float(eigs[-1]),
        'adj_min': float(eigs[0]),
        'adj_range': float(eigs[-1] - eigs[0]),
        'adj_sum_abs': float(np.sum(np.abs(eigs))),
    }

def invariantes_completos(G):
    """Recolecta todos los invariantes."""
    inv = {}
    inv.update(invariantes_basicos(G))
    inv.update(invariantes_espectrales(G))
    inv.update(invariantes_espectrales_normalizados(G))
    inv.update(caminos_mas_cortos(G))
    inv.update(autovalores_adyacencia(G))
    # Ciclos por longitud
    ciclos = ciclos_por_longitud(G)
    for L, n in ciclos.items():
        inv[f'cycles_L{L}'] = n
    inv['total_cycles'] = sum(ciclos.values())
    # Árboles de expansión
    try:
        inv['spanning_trees'] = spanning_trees(G)
    except Exception:
        inv['spanning_trees'] = None
    return inv

# ============================================================
# BÚSQUEDA COMBINATORIA
# ============================================================
def buscar_ratio(inv_e, inv_p, objetivo, tolerancia_rel=0.01, max_factor=3):
    """
    Busca combinaciones de la forma (inv_p^a * inv_e^b) / (inv_p^c * inv_e^d)
    que den el objetivo, con exponentes enteros en [-max_factor, max_factor].
    """
    # Filtrar invariantes que existen en ambos
    keys_comunes = [k for k in inv_e.keys()
                    if k in inv_p and inv_e[k] is not None and inv_p[k] is not None]

    # Filtrar valores válidos
    keys_validas = []
    for k in keys_comunes:
        if isinstance(inv_e[k], (int, float)) and isinstance(inv_p[k], (int, float)):
            if inv_e[k] > 0 and inv_p[k] > 0:
                keys_validas.append(k)

    print(f"Invariantes válidos para búsqueda: {len(keys_validas)}")

    hits = []
    exponentes = list(range(-max_factor, max_factor+1))

    # Búsqueda: ratio = (inv_p[k1]^a) * (inv_e[k2]^b) / (inv_p[k3]^c * inv_e[k4]^d)
    for k1, k2, k3, k4 in itertools.combinations_with_replacement(keys_validas, 4):
        for a, b, c, d in itertools.product(exponentes, repeat=4):
            if a == 0 and b == 0 and c == 0 and d == 0:
                continue
            try:
                num = (inv_p[k1]**a) * (inv_e[k2]**b)
                den = (inv_p[k3]**c) * (inv_e[k4]**d)
                if den == 0 or not np.isfinite(num) or not np.isfinite(den):
                    continue
                valor = num / den
                err_rel = abs(valor - objetivo) / objetivo
                if err_rel < tolerancia_rel:
                    hits.append({
                        'formula': f"({k1}^({a}) * {k2}^({b})) / ({k3}^({c}) * {k4}^({d}))",
                        'valor': valor,
                        'error_rel': err_rel,
                    })
            except (OverflowError, ZeroDivisionError):
                continue

    # Ordenar por error
    hits.sort(key=lambda h: h['error_rel'])
    return hits

# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    print("=" * 70)
    print("AA SEARCH — BÚSQUEDA DE INVARIANTES TOPOLÓGICOS")
    print("=" * 70)

    print("\nConstruyendo clústeres...")
    G19 = build_cluster_19()
    G57 = build_cluster_57()

    print("\nCalculando invariantes del Clúster 19...")
    inv_19 = invariantes_completos(G19)

    print("Calculando invariantes del Hard-Lock 57...")
    inv_57 = invariantes_completos(G57)

    # Imprimir resumen
    print("\n--- INVARIANTES CLAVE ---")
    print(f"{'Invariante':<25} {'Clúster 19':<20} {'Hard-Lock 57':<20} {'Ratio 57/19'}")
    print("-" * 85)
    keys_relevantes = ['N', 'E', 'dim_cycle', 'total_cycles', 'lambda_1',
                       'lambda_max', 'spectral_gap', 'diametro',
                       'camino_promedio', 'spanning_trees']
    for k in keys_relevantes:
        if k in inv_19 and k in inv_57:
            ratio = inv_57[k] / inv_19[k] if inv_19[k] != 0 else float('inf')
            print(f"{k:<25} {inv_19[k]:<20.6g} {inv_57[k]:<20.6g} {ratio:.6g}")

    # Búsqueda para m_p/m_e
    print("\n" + "=" * 70)
    print("BÚSQUEDA PARA m_p/m_e = 1836.15")
    print("=" * 70)
    hits = buscar_ratio(inv_19, inv_57, 1836.152673426,
                       tolerancia_rel=0.001, max_factor=2)

    print(f"\nEncontrados {len(hits)} candidatos con error < 0.1%:")
    for i, h in enumerate(hits[:20], 1):
        print(f"  {i}. {h['formula']}")
        print(f"     Valor: {h['valor']:.6g}   Error: {h['error_rel']*100:.4f}%")